# Chinook Database — Análise de Dados com SQL

## 1. Introdução

### 1.1 Contexto e base de dados
Este projeto utiliza o banco de dados *Chinook*, que simula as operações de uma loja de música digital. O modelo contém dados de artistas, álbuns, faixas, clientes e transações comerciais ao redor do mundo. O foco central é realizar uma Análise Exploratória de Dados (EDA) e responder a problemas de negócio através de SQL.

### 1.2 Perguntas de negócio
- Quais países possuem o maior faturamento total e como se distribui o volume de vendas entre eles?
- Quais são os gêneros musicais mais rentáveis da plataforma?
- Quem são os clientes "Top Spenders" (que mais gastaram) e qual o ticket médio por compra?
- Existem artistas ou álbuns que lideram as vendas de forma isolada?

### 1.3 Configuração do ambiente e conexão com o banco de dados

In [24]:
import sqlite3
import pandas as pd

path = "C:/Users/gessy/Documents/sql_portifolio/chinook/Data/chinook.sqlite"
conn = sqlite3.connect(path)

df_artists = pd.read_sql("SELECT * FROM Artist LIMIT 10", conn)
df_artists.head()

,ArtistId,Name
0,1,AC/DC
1,2,Accept
2,3,Aerosmith
3,4,Alanis Morissette
4,5,Alice In Chains


## 2. Análise de faturamento por país
Nesta etapa, vamos identificar quais mercados são mais lucrativos para a loja. Para isso, consultaremos a tabela *Invoice* (Faturas), agrupando os valores totais por país de cobrança.

In [31]:
query_vendas_pais = """
SELECT 
    BillingCountry AS Pais, 
    ROUND(SUM(Total), 2) AS Faturamento_Total,
    COUNT(InvoiceId) AS Qtd_Vendas,
    ROUND(SUM(Total) / COUNT(InvoiceId), 2) AS Ticket_Medio
FROM Invoice
GROUP BY Pais
ORDER BY Faturamento_Total DESC
LIMIT 10
"""

df_vendas_pais = pd.read_sql(query_vendas_pais, conn)
df_vendas_pais.index = df_vendas_pais.index + 1
df_vendas_pais

,Pais,Faturamento_Total,Qtd_Vendas,Ticket_Medio
1,USA,523.06,91,5.75
2,Canada,303.96,56,5.43
3,France,195.10,35,5.57
4,Brazil,190.10,35,5.43
5,Germany,156.48,28,5.59
6,United Kingdom,112.86,21,5.37
7,Czech Republic,90.24,14,6.45
8,Portugal,77.24,14,5.52
9,India,75.26,13,5.79
10,Chile,46.62,7,6.66


> Ao observar o *Ticket Médio*, você pode notar que alguns países podem ter menos vendas totais, mas os clientes gastam mais em cada compra.

## 3. Análise de preferência por gênero musical
Nesta seção, identificamos quais estilos musicais dominam as vendas na plataforma.

In [30]:
query_generos = """
SELECT 
    g.Name AS Genero, 
    SUM(il.Quantity) AS Total_Vendido
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY Genero
ORDER BY Total_Vendido DESC
LIMIT 5
"""
df_generos = pd.read_sql(query_generos, conn)
df_generos.index = df_generos.index + 1

df_generos

,Genero,Total_Vendido
1,Rock,835
2,Latin,386
3,Metal,264
4,Alternative & Punk,244
5,Jazz,80


> O gênero Rock é o líder em volume de vendas, representando a maior parte das transações.

## 4. Análise de clientes (Top Spenders)
Nesta seção, identificamos os clientes que geraram o maior volume de compras.

In [36]:
query_top_clientes = """
SELECT 
    c.FirstName || ' ' || c.LastName AS Nome_Cliente,
    c.Country AS Pais,
    ROUND(SUM(i.Total), 2) AS Gasto_Total
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY 1
ORDER BY Gasto_Total DESC
LIMIT 10
"""

df_top_clientes = pd.read_sql(query_top_clientes, conn)
df_top_clientes.index = df_top_clientes.index + 1

df_top_clientes

,Nome_Cliente,Pais,Gasto_Total
1,Helena Holý,Czech Republic,49.62
2,Richard Cunningham,USA,47.62
3,Luis Rojas,Chile,46.62
4,Ladislav Kovács,Hungary,45.62
5,Hugh O'Reilly,Ireland,45.62
6,Julia Barnett,USA,43.62
7,Fynn Zimmermann,Germany,43.62
8,Frank Ralston,USA,43.62
9,Victor Stevens,USA,42.62
10,Astrid Gruber,Austria,42.62


## 5. Análise de Artistas e Álbuns
Nesta seção, buscamos identificar se existe uma concentração de vendas em artistas específicos ou se o consumo é distribuído.

In [37]:
query_top_artistas = """
SELECT 
    ar.Name AS Artista, 
    SUM(il.Quantity) AS Total_Vendido,
    ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS Receita_Gerada
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
JOIN Track t ON al.AlbumId = t.AlbumId
JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY 1
ORDER BY Total_Vendido DESC
LIMIT 10
"""
df_top_artistas = pd.read_sql(query_top_artistas, conn)
df_top_artistas.index = df_top_artistas.index + 1

df_top_artistas

,Artista,Total_Vendido,Receita_Gerada
1,Iron Maiden,140,138.60
2,U2,107,105.93
3,Metallica,91,90.09
4,Led Zeppelin,87,86.13
5,Os Paralamas Do Sucesso,45,44.55
6,Deep Purple,44,43.56
7,Faith No More,42,41.58
8,Lost,41,81.59
9,Eric Clapton,40,39.60
10,R.E.M.,39,38.61


> A liderança de bandas de Rock no ranking de artistas reforça a análise anterior de gêneros.

## 6. Conclusões
Após a análise exploratória do banco de dados Chinook, os seguintes pontos se destacam para a estratégia do negócio:

- **Dominância do Rock:** O gênero Rock não é apenas o mais vendido, mas também o que possui os artistas mais rentáveis (como Iron Maiden e U2). Isso sugere que a base de clientes é composta majoritariamente por fãs de clássicos, o que direciona o marketing para lançamentos e promoções nesse segmento.

- **Concentração Geográfica:** Os EUA são o mercado principal, mas países como Canadá e Brasil apresentam um volume de vendas relevante. O ticket médio consistente entre os top 10 países indica que a precificação está bem equilibrada globalmente.

- **Fidelização:** Identificamos os clientes com maior gasto acumulado. Para esses usuários, a loja poderia implementar programas de fidelidade ou ofertas exclusivas, garantindo a retenção da fatia mais lucrativa da base.

- **Catálogo:** A forte presença de bandas com catálogos extensos no topo do ranking mostra que álbuns completos ainda possuem grande peso nas vendas, apesar da tendência de consumo de faixas isoladas.

In [39]:
conn.close()